# BigAlpha 2026 volatility_5 neg

AI-assisted official factorlib single-column baseline. Source column=volatility_5, sign=-1.0, template ModelScore=1715426.2465.


In [ ]:
from __future__ import annotations


FACTOR_COLUMN = 'volatility_5'
FACTOR_SIGN = -1.0


def _zscore(series):
    import numpy as np
    import pandas as pd

    x = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    if x.notna().sum() < 5:
        return x * 0.0
    x = x.clip(x.quantile(0.01), x.quantile(0.99))
    std = x.std(ddof=0)
    if not std or pd.isna(std):
        return x * 0.0
    return (x - x.mean()) / std


def _pool(dai, start_ts, end_ts):
    import pandas as pd

    try:
        pool = dai.query(
            "SELECT date, instrument::string AS instrument FROM bigalpha_2026_instruments",
            filters={"date": [f"{start_ts:%Y-%m-%d} 00:00:00", f"{end_ts:%Y-%m-%d} 23:59:59"]},
            compression=True,
        ).df()
    except Exception:
        return pd.DataFrame(columns=["date", "instrument"])
    if pool.empty:
        return pool.reindex(columns=["date", "instrument"])
    pool["date"] = pd.to_datetime(pool["date"]).dt.normalize()
    pool["instrument"] = pool["instrument"].astype(str)
    return pool.drop_duplicates(["date", "instrument"], keep="last")


def main(datasources, start_date, end_date):
    import numpy as np
    import pandas as pd
    import dai

    start_ts = pd.Timestamp(start_date).normalize()
    end_ts = pd.Timestamp(end_date).normalize()

    sql = f"""
    SELECT
        date,
        instrument::string AS instrument,
        {FACTOR_COLUMN} AS raw_factor
    FROM bigalpha_2026_factorlib
    """
    try:
        df = dai.query(
            sql,
            filters={"date": [f"{start_ts:%Y-%m-%d} 00:00:00", f"{end_ts:%Y-%m-%d} 23:59:59"]},
            compression=True,
        ).df()
    except Exception:
        pool = _pool(dai, start_ts, end_ts)
        pool["factor"] = 0.0
        return pool.reindex(columns=["date", "instrument", "factor"]).sort_values(
            ["date", "instrument"]
        ).reset_index(drop=True)

    if df.empty:
        pool = _pool(dai, start_ts, end_ts)
        pool["factor"] = 0.0
        return pool.reindex(columns=["date", "instrument", "factor"]).sort_values(
            ["date", "instrument"]
        ).reset_index(drop=True)

    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype(str)
    df["raw_factor"] = pd.to_numeric(df["raw_factor"], errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    )
    df["factor"] = FACTOR_SIGN * df.groupby("date", observed=True)["raw_factor"].transform(_zscore)
    out = df[["date", "instrument", "factor"]].drop_duplicates(
        ["date", "instrument"], keep="last"
    )

    pool = _pool(dai, start_ts, end_ts)
    if not pool.empty:
        out = pool.merge(out, on=["date", "instrument"], how="left")

    out["factor"] = pd.to_numeric(out["factor"], errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    )
    out["factor"] = out.groupby("date", observed=True)["factor"].transform(
        lambda s: s.fillna(s.median())
    )
    out["factor"] = out["factor"].fillna(0.0).astype("float32")
    return out[["date", "instrument", "factor"]].sort_values(
        ["date", "instrument"]
    ).reset_index(drop=True)
